# Shared TX + Multilingual Receiver Test

This notebook evaluates the synchronized BPE DeepSC model using **one shared transmission** per batch:

```text
English source
    ↓
Shared Semantic Encoder
    ↓
Channel Encoder
    ↓
Wireless Channel
    ↓
Rx_sig  ← generated once
    ├── <EN> → English
    ├── <PT> → Portuguese
    ├── <ES> → Spanish
    └── <FR> → French
```

The same `Rx_sig` is reused for every requested target language.

The notebook supports:
- `val` or `test` split;
- greedy or beam decoding;
- multiple SNR values;
- repeated channel trials;
- BLEU, chrF++, semantic similarity, ROUGE-L, exact match, and BPE token accuracy;
- CSV and JSON export.

For a quick check, start with `SPLIT = "val"`, `MAX_SAMPLES = 20`, and `METRICS = False`.


## 1. Environment

Run this notebook from the **DeepSC repository root** so imports such as `dataset_multilingual`, `models`, and `utils` resolve correctly.

If any evaluation packages are missing, install them in your project environment before continuing.


In [ ]:
# Optional dependency installation
# Uncomment only if needed.
#
# %pip install sentencepiece sacrebleu rouge-score sentence-transformers pandas tqdm


## 2. Imports

In [1]:
from __future__ import annotations

import csv
import json
import os
import random
from functools import partial
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, Iterable, List, Sequence

import numpy as np
import pandas as pd
import sacrebleu
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset_multilingual import (
    SUPPORTED_LANGUAGES,
    EurMultilingualDatasetBPE,
    collate_multilingual_bpe,
)
from metrics.metrics_bpe import DeepSCMetrics, normalize_text as normalize_metric_text
from models.transceiver import DeepSC
from utils.bpe_utils import SeqtoTextBPE, beam_decode, greedy_decode
from utils.model_utils import power_normalize, send_through_channel

LANGUAGE_TOKENS = {
    "en": "<EN>",
    "pt": "<PT>",
    "es": "<ES>",
    "fr": "<FR>",
}


/home/prinako/External/WorkSpace/PGCC/DeepSC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuration

Edit the checkpoint paths below before running the notebook.

For development/smoke testing:
- use `SPLIT = "val"`;
- keep `MAX_SAMPLES` small;
- optionally set `METRICS = False`.

For the final experiment:
- set `SPLIT = "test"`;
- set `MAX_SAMPLES = None`;
- enable metrics;
- use the final selected checkpoint.


In [14]:
# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
DATA_DIR = "data/train/europarl_bpe/2026-09-19"

ENCODER_CHECKPOINT = "results/base_model/2026-09-19/encoder_17.pth"
DECODER_CHECKPOINT = "results/base_model/2026-09-19/decoder_17.pth"

# Optional. If None, the notebook looks for run_config.json in
# the checkpoint directory.
RUN_CONFIG = None

OUTPUT_DIR = "results/multilingual_test_notebook"

# ------------------------------------------------------------------
# Evaluation protocol
# ------------------------------------------------------------------
SPLIT = "val"                 # "val" for smoke test; "test" for final evaluation
LANGUAGES = ("en", "pt", "es", "fr")

CHANNEL = "Rayleigh"
SNR_DB = [0,1,3,5,6,8.0, 10, 12, 14, 16, 18]                # Example final grid: [0, 4, 8, 12, 16]
TRIALS = 2

DECODE = "greedy"             # "greedy" or "beam"
BEAM_SIZE = 5
LENGTH_PENALTY = 0.7

BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 48

MAX_SAMPLES = 20              # None = full split

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------
METRICS = True               # Turn on after the smoke test
SEMANTIC_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
SEMANTIC_BATCH_SIZE = 256

# Diagnostic override only.
ALLOW_UNTRAINED_LANGUAGES = False

# Optional architecture overrides.
# Leave as None to read from run_config/preprocessing configuration.
MAX_LENGTH = None
NUM_LAYERS = None
NUM_HEADS = None
D_MODEL = None
DFF = None
DROPOUT = None


## 4. Helper functions

In [4]:
def setup_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def resolve_data_dir(path: str) -> str:
    root = Path(path)

    required_names = (
        "train_multilingual.pkl",
        "val_multilingual.pkl",
        "test_multilingual.pkl",
        "vocab_bpe.json",
        "tokenizer_bpe.model",
        "preprocessing_config.json",
    )

    if root.is_dir() and all((root / name).exists() for name in required_names):
        return str(root)

    if not root.exists():
        raise FileNotFoundError(f"Dataset directory does not exist: {root}")

    candidates = []
    for child in root.iterdir():
        if child.is_dir() and all((child / name).exists() for name in required_names):
            candidates.append(child)

    if not candidates:
        raise FileNotFoundError(
            "Could not find a complete synchronized multilingual preprocessing "
            f"output under: {root}"
        )

    resolved = sorted(candidates, key=lambda item: item.name)[-1]
    print(f"Using latest preprocessing directory: {resolved}")
    return str(resolved)


def parse_languages(values: Sequence[str]) -> tuple[str, ...]:
    languages = tuple(str(value).lower() for value in values)

    unsupported = set(languages).difference(SUPPORTED_LANGUAGES)
    if unsupported:
        raise ValueError(
            f"Unsupported languages: {sorted(unsupported)}. "
            f"Supported: {list(SUPPORTED_LANGUAGES)}"
        )

    if len(languages) != len(set(languages)):
        raise ValueError("Duplicate target languages are not allowed.")

    if not languages:
        raise ValueError("At least one target language is required.")

    return languages


def resolve_run_config(
    encoder_checkpoint: str,
    decoder_checkpoint: str,
    run_config_path: str | None,
) -> Dict:
    if run_config_path:
        path = Path(run_config_path)
    else:
        enc_parent = Path(encoder_checkpoint).resolve().parent
        dec_parent = Path(decoder_checkpoint).resolve().parent

        if enc_parent != dec_parent:
            print("[Warning] encoder/decoder checkpoint directories differ.")
            return {}

        path = enc_parent / "run_config.json"

    if not path.exists():
        print("[Warning] run_config.json not found.")
        return {}

    print(f"Teacher run config: {path}")
    return load_json(str(path))


def choose_model_value(cli_value, run_config: Dict, key: str, fallback):
    if cli_value is not None:
        return cli_value

    if key in run_config and run_config[key] is not None:
        return run_config[key]

    return fallback


def build_model(
    num_vocab: int,
    preprocessing_config: Dict,
    run_config: Dict,
    device: torch.device,
):
    preprocessing_max_len = int(preprocessing_config.get("max_seq_len", 64))

    max_len = int(
        choose_model_value(
            MAX_LENGTH,
            run_config,
            "max_length",
            preprocessing_max_len,
        )
    )
    num_layers = int(
        choose_model_value(NUM_LAYERS, run_config, "num_layers", 8)
    )
    num_heads = int(
        choose_model_value(NUM_HEADS, run_config, "num_heads", 16)
    )
    d_model = int(
        choose_model_value(D_MODEL, run_config, "d_model", 128)
    )
    dff = int(
        choose_model_value(DFF, run_config, "dff", 512)
    )
    dropout = float(
        choose_model_value(DROPOUT, run_config, "dropout", 0.1)
    )

    if max_len < preprocessing_max_len:
        raise ValueError(
            f"Model max_len={max_len} is smaller than preprocessing "
            f"max_seq_len={preprocessing_max_len}."
        )

    configured_vocab = run_config.get("vocab_size")
    if configured_vocab is not None and int(configured_vocab) != int(num_vocab):
        raise ValueError(
            "Teacher run_config vocabulary does not match the evaluation dataset: "
            f"teacher={configured_vocab}, dataset={num_vocab}."
        )

    model = DeepSC(
        num_layers,
        num_vocab,
        num_vocab,
        max_len,
        max_len,
        d_model,
        num_heads,
        dff,
        dropout,
    ).to(device)

    enc = torch.load(
        ENCODER_CHECKPOINT,
        map_location=device,
        weights_only=False,
    )
    dec = torch.load(
        DECODER_CHECKPOINT,
        map_location=device,
        weights_only=False,
    )

    required_enc = {"encoder", "channel_encoder"}
    required_dec = {"channel_decoder", "decoder", "dense"}

    if not required_enc.issubset(enc):
        raise KeyError(
            f"Encoder checkpoint missing keys: {sorted(required_enc.difference(enc))}"
        )

    if not required_dec.issubset(dec):
        raise KeyError(
            f"Decoder checkpoint missing keys: {sorted(required_dec.difference(dec))}"
        )

    model.encoder.load_state_dict(enc["encoder"], strict=True)
    model.channel_encoder.load_state_dict(enc["channel_encoder"], strict=True)
    model.channel_decoder.load_state_dict(dec["channel_decoder"], strict=True)
    model.decoder.load_state_dict(dec["decoder"], strict=True)
    model.dense.load_state_dict(dec["dense"], strict=True)
    model.eval()

    architecture = {
        "max_length": max_len,
        "num_layers": num_layers,
        "num_heads": num_heads,
        "d_model": d_model,
        "dff": dff,
        "dropout": dropout,
        "vocab_size": num_vocab,
    }

    return model, architecture


def validate_trained_languages(
    run_config: Dict,
    requested_languages: Sequence[str],
    allow_untrained_languages: bool,
) -> None:
    trained_languages = run_config.get("languages")

    if not trained_languages:
        print(
            "[Warning] run_config.json does not identify trained languages; "
            "language coverage cannot be verified."
        )
        return

    trained = {str(lang).lower() for lang in trained_languages}
    requested = set(requested_languages)
    missing = requested.difference(trained)

    if missing and not allow_untrained_languages:
        raise ValueError(
            "The checkpoint run_config does not show training for requested "
            f"languages {sorted(missing)}. Trained languages: {sorted(trained)}."
        )

    if missing:
        print(
            "[Warning] testing languages not listed in Teacher training: "
            + ", ".join(sorted(missing))
        )


def make_src_mask(
    src: torch.Tensor,
    pad_idx: int,
    device: torch.device,
) -> torch.Tensor:
    return (
        (src == pad_idx)
        .unsqueeze(-2)
        .to(device=device, dtype=torch.float32)
    )


@torch.no_grad()
def shared_transmission(
    model: DeepSC,
    src: torch.Tensor,
    src_mask: torch.Tensor,
    channel: str,
    snr_db: float,
):
    # Shared transmitter.
    enc_output = model.encoder(src, src_mask)
    channel_enc_output = model.channel_encoder(enc_output)
    tx_sig = power_normalize(channel_enc_output)

    # ONE wireless-channel realization.
    rx_sig = send_through_channel(tx_sig, channel, snr_db)

    return tx_sig, rx_sig


def decode_language(
    model: DeepSC,
    rx_sig: torch.Tensor,
    src: torch.Tensor,
    src_mask: torch.Tensor,
    target_language_idx: int,
    pad_idx: int,
    start_idx: int,
    end_idx: int,
    max_len: int,
    snr_db: float,
    device: torch.device,
    forbidden_token_ids: Iterable[int],
) -> torch.Tensor:
    common = dict(
        Rx_sig=rx_sig,
        model=model,
        src=src,
        src_mask=src_mask,
        max_len=max_len,
        start_idx=start_idx,
        padding_idx=pad_idx,
        end_idx=end_idx,
        target_language_idx=target_language_idx,
        channel=CHANNEL,
        snr=snr_db,
        device=device,
    )

    if DECODE == "greedy":
        return greedy_decode(**common)

    return beam_decode(
        **common,
        beam_size=BEAM_SIZE,
        length_penalty=LENGTH_PENALTY,
        forbidden_token_ids=forbidden_token_ids,
        return_all_beams=False,
    )


In [5]:
def scalable_metrics(
    evaluator: DeepSCMetrics,
    references: Sequence[str],
    hypotheses: Sequence[str],
    semantic_batch_size: int,
) -> Dict[str, float]:
    if len(references) != len(hypotheses):
        raise ValueError("references and hypotheses must have equal length.")

    if not references:
        raise ValueError("Cannot evaluate an empty corpus.")

    refs = [normalize_metric_text(x) for x in references]
    hyps = [normalize_metric_text(x) for x in hypotheses]

    bleu = float(sacrebleu.corpus_bleu(hyps, [refs]).score)
    chrf = float(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score)

    semantic_sum = 0.0
    semantic_count = 0

    for start in range(0, len(refs), semantic_batch_size):
        stop = min(start + semantic_batch_size, len(refs))

        ref_emb = evaluator.semantic_model.encode(
            refs[start:stop],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        hyp_emb = evaluator.semantic_model.encode(
            hyps[start:stop],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        scores = np.sum(ref_emb * hyp_emb, axis=1)
        semantic_sum += float(scores.sum())
        semantic_count += int(len(scores))

    rouge_values = [
        evaluator.rouge_l(ref, hyp)
        for ref, hyp in zip(refs, hyps)
    ]

    exact_values = [
        evaluator.exact_match(ref, hyp)
        for ref, hyp in zip(refs, hyps)
    ]

    token_values = [
        evaluator.token_accuracy(ref, hyp)
        for ref, hyp in zip(refs, hyps)
    ]

    return {
        "bleu": bleu,
        "chrf": chrf,
        "semantic_similarity": semantic_sum / max(semantic_count, 1),
        "rouge_l": float(np.mean(rouge_values)),
        "exact_match_rate": float(np.mean(exact_values)),
        "token_accuracy": float(np.mean(token_values)),
        "num_samples": len(refs),
    }


def aggregate_trial_metrics(per_trial: Sequence[Dict]) -> pd.DataFrame:
    metric_names = (
        "bleu",
        "chrf",
        "semantic_similarity",
        "rouge_l",
        "exact_match_rate",
        "token_accuracy",
    )

    grouped: Dict[tuple[float, str], List[Dict]] = {}

    for row in per_trial:
        key = (float(row["snr_db"]), str(row["language"]))
        grouped.setdefault(key, []).append(row)

    result = []

    for (snr_db, language), rows in sorted(grouped.items()):
        item = {
            "snr_db": snr_db,
            "language": language,
            "num_trials": len(rows),
            "num_samples_per_trial": rows[0].get("num_samples", 0),
        }

        for metric in metric_names:
            values = np.asarray(
                [float(row[metric]) for row in rows],
                dtype=float,
            )
            item[f"{metric}_mean"] = float(values.mean())
            item[f"{metric}_std"] = (
                float(values.std(ddof=1))
                if len(values) > 1
                else 0.0
            )

        result.append(item)

    return pd.DataFrame(result)


## 5. Load dataset, tokenizer, and checkpoint

In [6]:
LANGUAGES = parse_languages(LANGUAGES)
setup_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resolved_data_dir = resolve_data_dir(DATA_DIR)

vocab_path = os.path.join(resolved_data_dir, "vocab_bpe.json")
tokenizer_path = os.path.join(resolved_data_dir, "tokenizer_bpe.model")
preprocessing_path = os.path.join(
    resolved_data_dir,
    "preprocessing_config.json",
)

vocab = load_json(vocab_path)
preprocessing_config = load_json(preprocessing_path)

run_config = resolve_run_config(
    ENCODER_CHECKPOINT,
    DECODER_CHECKPOINT,
    RUN_CONFIG,
)

token_to_idx = vocab["token_to_idx"]
num_vocab = int(vocab["vocab_size"])

pad_idx = int(token_to_idx["<PAD>"])
start_idx = int(token_to_idx["<START>"])
end_idx = int(token_to_idx["<END>"])

language_ids = {
    lang: int(token_to_idx[LANGUAGE_TOKENS[lang]])
    for lang in LANGUAGES
}

validate_trained_languages(
    run_config,
    LANGUAGES,
    ALLOW_UNTRAINED_LANGUAGES,
)

model, architecture = build_model(
    num_vocab,
    preprocessing_config,
    run_config,
    device,
)

max_len = int(architecture["max_length"])

dataset = EurMultilingualDatasetBPE(
    split=SPLIT,
    data_dir=resolved_data_dir,
    languages=LANGUAGES,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=partial(
        collate_multilingual_bpe,
        pad_idx=pad_idx,
    ),
)

converter = SeqtoTextBPE(
    vocb_dictionary=token_to_idx,
    end_idx=end_idx,
    tokenizer_model=tokenizer_path,
)

evaluator = None

if METRICS:
    evaluator = DeepSCMetrics(
        sp=converter.tokenizer,
        semantic_model=SEMANTIC_MODEL,
        device=str(device),
    )

all_control_ids = {
    pad_idx,
    start_idx,
    *[
        int(token_to_idx[token])
        for token in LANGUAGE_TOKENS.values()
        if token in token_to_idx
    ],
}

print("=" * 72)
print("SHARED-TX MULTILINGUAL NOTEBOOK")
print("=" * 72)
print(f"Device              : {device}")
print(f"Dataset             : {resolved_data_dir}")
print(f"Split               : {SPLIT}")
print(f"Samples             : {len(dataset):,}")
print(f"Languages           : {', '.join(x.upper() for x in LANGUAGES)}")
print(f"SNRs                : {SNR_DB}")
print(f"Channel             : {CHANNEL}")
print(f"Decode              : {DECODE}")
print(f"Max length          : {max_len}")
print(f"Vocabulary          : {num_vocab:,}")
print("Shared transmission : ON")
print("=" * 72)

architecture


Teacher run config: /home/prinako/External/WorkSpace/PGCC/DeepSC/results/base_model/2026-09-19/run_config.json


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 857.35it/s]


SHARED-TX MULTILINGUAL NOTEBOOK
Device              : cuda:0
Dataset             : data/train/europarl_bpe/2026-09-19
Split               : val
Samples             : 231,881
Languages           : EN, PT, ES, FR
SNRs                : [8.0]
Channel             : Rayleigh
Decode              : greedy
Max length          : 64
Vocabulary          : 32,000
Shared transmission : ON


{'max_length': 64,
 'num_layers': 8,
 'num_heads': 16,
 'd_model': 128,
 'dff': 512,
 'dropout': 0.1,
 'vocab_size': 32000}

## 6. Inspect one sample before transmission

This checks that:
- TX input is the shared English sequence;
- each target starts with its own language-control token;
- all four targets belong to the same aligned sample.


In [7]:
sample = dataset[0]

print("ID:", sample["id"])
print("TX:", converter.sequence_to_text(sample["src"]))

for lang in LANGUAGES:
    print(
        f"{lang.upper()} target:",
        converter.sequence_to_text(sample["targets"][lang]),
    )


ID: 312325
TX: to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.
EN target: to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.
PT target: fazer isso seria como dizer a alguém decidido a assassinar: 'por favor, não cometa assassinato, mas se o fizer, não haverá qualquer processo criminal'.
ES target: hacerlo sería como decirle a alguien decidido a asesinar, 'por favor, no cometa asesinato, pero si lo hace, no habrá ningún procedimiento penal de todos modos'.
FR target: ce serait comme dire à quelqu'un déterminé à tuer: "s'il vous plaît, ne commettez pas de meurtre, mais si vous le faites, il n'y aura pas de procédure pénale de toute façon".


## 7. One shared transmission → four receiver outputs

This cell demonstrates the central property directly.

`shared_transmission(...)` is called **once**.  
The returned `rx_sig` is then reused for EN, PT, ES, and FR decoding.


In [8]:
demo_batch = next(iter(loader))

src = demo_batch["src"][:1].to(device)
src_mask = make_src_mask(src, pad_idx, device)

demo_snr = float(SNR_DB[0])

# ------------------------------------------------------------
# ONE shared transmission.
# ------------------------------------------------------------
tx_sig, rx_sig = shared_transmission(
    model=model,
    src=src,
    src_mask=src_mask,
    channel=CHANNEL,
    snr_db=demo_snr,
)

print("TX tensor shape:", tuple(tx_sig.shape))
print("RX tensor shape:", tuple(rx_sig.shape))
print()
print("Original EN:", converter.sequence_to_text(src[0].cpu()))
print("-" * 72)

demo_results = []

for lang in LANGUAGES:
    generated = decode_language(
        model=model,
        rx_sig=rx_sig,                    # SAME rx_sig for every language
        src=src,
        src_mask=src_mask,
        target_language_idx=language_ids[lang],
        pad_idx=pad_idx,
        start_idx=start_idx,
        end_idx=end_idx,
        max_len=max_len,
        snr_db=demo_snr,
        device=device,
        forbidden_token_ids=all_control_ids,
    )

    prediction = converter.sequence_to_text(
        generated[0].detach().cpu()
    )

    reference = converter.sequence_to_text(
        demo_batch["targets"][lang][0]
    )

    demo_results.append(
        {
            "language": lang.upper(),
            "reference": reference,
            "prediction": prediction,
        }
    )

pd.DataFrame(demo_results)


TX tensor shape: (1, 38, 16)
RX tensor shape: (1, 38, 16)

Original EN: to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.
------------------------------------------------------------------------


,language,reference,prediction
0,EN,to do so would be like saying to someone bent ...,to do so would be like to say to benting to ma...
1,PT,fazer isso seria como dizer a alguém decidido ...,"para que seria, por dizer-se de cerca de bem, ..."
2,ES,hacerlo sería como decirle a alguien decidido ...,"para que se puede decir que se refiere a la ""a..."
3,FR,ce serait comme dire à quelqu'un déterminé à t...,"pour le faire, je voudrais dire à dire d'un ""s..."


## 8. Full evaluation loop

Each batch/SNR/trial does:

1. shared English TX;
2. one channel realization;
3. one `rx_sig`;
4. reuse that `rx_sig` for every requested language.

The final test split should only be used after the checkpoint and evaluation protocol are fixed.


In [9]:
prediction_rows: List[Dict] = []
per_trial_metrics: List[Dict] = []

for snr_db in SNR_DB:
    for trial in range(TRIALS):
        trial_seed = (
            SEED
            + int(round(float(snr_db) * 1000))
            + trial * 1_000_003
        )
        setup_seed(trial_seed)

        collected = {
            lang: {
                "references": [],
                "hypotheses": [],
            }
            for lang in LANGUAGES
        }

        processed = 0

        pbar = tqdm(
            loader,
            desc=f"SNR={snr_db:g} dB trial={trial + 1}",
        )

        for batch in pbar:
            remaining = None

            if MAX_SAMPLES is not None:
                remaining = MAX_SAMPLES - processed
                if remaining <= 0:
                    break

            ids = batch["ids"]
            src = batch["src"]
            targets = batch["targets"]

            if remaining is not None and src.size(0) > remaining:
                src = src[:remaining]
                ids = ids[:remaining]
                targets = {
                    lang: tensor[:remaining]
                    for lang, tensor in targets.items()
                }

            src = src.to(device, non_blocking=True)
            src_mask = make_src_mask(
                src,
                pad_idx,
                device,
            )

            # ============================================================
            # ONE shared TX + ONE wireless-channel realization.
            # ============================================================
            _, rx_sig = shared_transmission(
                model=model,
                src=src,
                src_mask=src_mask,
                channel=CHANNEL,
                snr_db=float(snr_db),
            )

            source_texts = [
                converter.sequence_to_text(row)
                for row in src.detach().cpu()
            ]

            # ============================================================
            # Language-controlled decoding from the SAME rx_sig.
            # ============================================================
            for lang in LANGUAGES:
                generated = decode_language(
                    model=model,
                    rx_sig=rx_sig,
                    src=src,
                    src_mask=src_mask,
                    target_language_idx=language_ids[lang],
                    pad_idx=pad_idx,
                    start_idx=start_idx,
                    end_idx=end_idx,
                    max_len=max_len,
                    snr_db=float(snr_db),
                    device=device,
                    forbidden_token_ids=all_control_ids,
                )

                predictions = [
                    converter.sequence_to_text(row)
                    for row in generated.detach().cpu()
                ]

                references = [
                    converter.sequence_to_text(row)
                    for row in targets[lang]
                ]

                collected[lang]["references"].extend(
                    references
                )
                collected[lang]["hypotheses"].extend(
                    predictions
                )

                for sample_id, source_text, reference, prediction in zip(
                    ids,
                    source_texts,
                    references,
                    predictions,
                ):
                    prediction_rows.append(
                        {
                            "id": sample_id,
                            "snr_db": float(snr_db),
                            "trial": trial + 1,
                            "channel": CHANNEL,
                            "decode": DECODE,
                            "source_en": source_text,
                            "target_language": lang,
                            "reference": reference,
                            "prediction": prediction,
                        }
                    )

            processed += src.size(0)
            pbar.set_postfix(samples=processed)

            if MAX_SAMPLES is not None and processed >= MAX_SAMPLES:
                break

        if METRICS:
            assert evaluator is not None

            for lang in LANGUAGES:
                metric_values = scalable_metrics(
                    evaluator,
                    collected[lang]["references"],
                    collected[lang]["hypotheses"],
                    semantic_batch_size=SEMANTIC_BATCH_SIZE,
                )

                row = {
                    "snr_db": float(snr_db),
                    "trial": trial + 1,
                    "language": lang,
                    **metric_values,
                }

                per_trial_metrics.append(row)

                print(
                    f"SNR={snr_db:g} "
                    f"trial={trial + 1} "
                    f"{lang.upper()} | "
                    f"BLEU={metric_values['bleu']:.2f} "
                    f"chrF={metric_values['chrf']:.2f} "
                    f"SEM={metric_values['semantic_similarity']:.4f}"
                )

predictions_df = pd.DataFrame(prediction_rows)

print(f"Prediction rows: {len(predictions_df):,}")
predictions_df.head(12)


SNR=8 dB trial=1:   0%|          | 0/7247 [00:00<?, ?it/s][W921 19:15:59.835039743 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 115343360 bytes (free: 46530560, total: 4007591936).
[W921 19:16:00.070558948 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 159383552 bytes (free: 75890688, total: 4007591936).
SNR=8 dB trial=1:   0%|          | 0/7247 [00:01<?, ?it/s, samples=20]


SNR=8 trial=1 EN | BLEU=66.52 chrF=73.16 SEM=0.7675
SNR=8 trial=1 PT | BLEU=21.48 chrF=47.78 SEM=0.6595
SNR=8 trial=1 ES | BLEU=22.67 chrF=47.49 SEM=0.6886
SNR=8 trial=1 FR | BLEU=21.10 chrF=44.45 SEM=0.6673
Prediction rows: 80


,id,snr_db,trial,channel,decode,source_en,target_language,reference,prediction
0,312325,8.0,1,Rayleigh,greedy,to do so would be like saying to someone bent ...,en,to do so would be like saying to someone bent ...,to do so would be like to say to benting at sp...
1,1354636,8.0,1,Rayleigh,greedy,it is my belief that we need a special represe...,en,it is my belief that we need a special represe...,it is my independence that we need a special r...
2,1004555,8.0,1,Rayleigh,greedy,with regard to the research framework programm...,en,with regard to the research framework programm...,with regard to the research framework programm...
3,1388064,8.0,1,Rayleigh,greedy,emu also rules out an economy that is sustaina...,en,emu also rules out an economy that is sustaina...,austria also rules out an economy that is sust...
4,889116,8.0,1,Rayleigh,greedy,"this fund, as you, commissioner, and my fellow...",en,"this fund, as you, commissioner, and my fellow...","this fund, as you, commissioner, and my fellow..."
5,996197,8.0,1,Rayleigh,greedy,this is why the 'women in science' research pr...,en,this is why the 'women in science' research pr...,this is why the 'w. in corruption's research p...
6,259151,8.0,1,Rayleigh,greedy,the issue of a secure long-term energy supply ...,en,the issue of a secure long-term energy supply ...,the issue of a long-term-term energy supply fo...
7,531406,8.0,1,Rayleigh,greedy,rural villages quickly become dormitory towns ...,en,rural villages quickly become dormitory towns ...,rural parents always become a aby families wit...
8,1305365,8.0,1,Rayleigh,greedy,item 12 about the community system on fisherie...,en,item 12 about the community system on fisherie...,point 12 about the community system on fisheri...
9,1110804,8.0,1,Rayleigh,greedy,"nevertheless, member states executed only 2 25...",en,"nevertheless, member states executed only 2 25...","nevertheless, member states out only only 2 25..."


## 9. Metric summary

In [10]:
if METRICS and per_trial_metrics:
    trial_metrics_df = pd.DataFrame(per_trial_metrics)
    aggregate_metrics_df = aggregate_trial_metrics(per_trial_metrics)

    display(trial_metrics_df)
    display(aggregate_metrics_df)
else:
    trial_metrics_df = pd.DataFrame()
    aggregate_metrics_df = pd.DataFrame()

    print(
        "Metrics are disabled. "
        "Set METRICS = True in the configuration cell and rerun."
    )


,snr_db,trial,language,bleu,chrf,semantic_similarity,rouge_l,exact_match_rate,token_accuracy,num_samples
0,8.0,1,en,66.517073,73.163776,0.767529,0.808860,0.15,0.670357,20
1,8.0,1,pt,21.481292,47.775066,0.659519,0.487211,0.00,0.198161,20
2,8.0,1,es,22.672144,47.492515,0.688579,0.505307,0.00,0.206071,20
3,8.0,1,fr,21.104087,44.449325,0.667273,0.468201,0.00,0.158951,20


,snr_db,language,num_trials,num_samples_per_trial,bleu_mean,bleu_std,chrf_mean,chrf_std,semantic_similarity_mean,semantic_similarity_std,rouge_l_mean,rouge_l_std,exact_match_rate_mean,exact_match_rate_std,token_accuracy_mean,token_accuracy_std
0,8.0,en,1,20,66.517073,0.0,73.163776,0.0,0.767529,0.0,0.808860,0.0,0.15,0.0,0.670357,0.0
1,8.0,es,1,20,22.672144,0.0,47.492515,0.0,0.688579,0.0,0.505307,0.0,0.00,0.0,0.206071,0.0
2,8.0,fr,1,20,21.104087,0.0,44.449325,0.0,0.667273,0.0,0.468201,0.0,0.00,0.0,0.158951,0.0
3,8.0,pt,1,20,21.481292,0.0,47.775066,0.0,0.659519,0.0,0.487211,0.0,0.00,0.0,0.198161,0.0


## 10. Inspect qualitative outputs

This is useful for checking the same source sentence across EN/PT/ES/FR.


In [11]:
if not predictions_df.empty:
    first_id = predictions_df.iloc[0]["id"]

    qualitative = predictions_df[
        predictions_df["id"] == first_id
    ][
        [
            "id",
            "snr_db",
            "trial",
            "source_en",
            "target_language",
            "reference",
            "prediction",
        ]
    ]

    display(qualitative)


,id,snr_db,trial,source_en,target_language,reference,prediction
0,312325,8.0,1,to do so would be like saying to someone bent ...,en,to do so would be like saying to someone bent ...,to do so would be like to say to benting at sp...
20,312325,8.0,1,to do so would be like saying to someone bent ...,pt,fazer isso seria como dizer a alguém decidido ...,"para fazer isso seria dizer que, em vez, ""aar-..."
40,312325,8.0,1,to do so would be like saying to someone bent ...,es,hacerlo sería como decirle a alguien decidido ...,"para hacerlo, quisiera decir que en el sr. ben..."
60,312325,8.0,1,to do so would be like saying to someone bent ...,fr,ce serait comme dire à quelqu'un déterminé à t...,"pour faire, c'est-à-dire à dire d'un ""un ""s, ""..."


## 11. Save CSV and JSON results

In [12]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

predictions_path = os.path.join(
    OUTPUT_DIR,
    "multilingual_predictions.csv",
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "summary.json",
)

predictions_df.to_csv(
    predictions_path,
    index=False,
    encoding="utf-8",
)

aggregate_records = (
    aggregate_metrics_df.to_dict(orient="records")
    if not aggregate_metrics_df.empty
    else []
)

summary = {
    "data_dir": resolved_data_dir,
    "split": SPLIT,
    "encoder_checkpoint": ENCODER_CHECKPOINT,
    "decoder_checkpoint": DECODER_CHECKPOINT,
    "architecture": architecture,
    "channel": CHANNEL,
    "snr_db": [float(x) for x in SNR_DB],
    "trials": TRIALS,
    "seed": SEED,
    "decode": DECODE,
    "beam_size": BEAM_SIZE if DECODE == "beam" else None,
    "length_penalty": LENGTH_PENALTY if DECODE == "beam" else None,
    "languages": list(LANGUAGES),
    "max_samples": MAX_SAMPLES,
    "metrics_enabled": bool(METRICS),
    "semantic_model": SEMANTIC_MODEL if METRICS else None,
    "shared_transmission": True,
    "prediction_rows": int(len(predictions_df)),
    "per_trial_metrics": per_trial_metrics,
    "aggregate_metrics": aggregate_records,
}

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:")
print(" ", predictions_path)
print(" ", summary_path)


Saved:
  results/multilingual_test_notebook/multilingual_predictions.csv
  results/multilingual_test_notebook/summary.json


## Final experiment checklist

Before using the numbers in a paper:

- [ ] final Teacher checkpoint is fixed;
- [ ] tokenizer and dataset match the checkpoint;
- [ ] `SPLIT = "test"` only after model selection is complete;
- [ ] SNR grid is fixed in advance;
- [ ] Teacher/CE/KD comparisons use matched channel draws;
- [ ] decoding strategy is identical across compared models;
- [ ] corpus BLEU is reported as corpus BLEU, not mean sentence BLEU;
- [ ] semantic-model version/revision is recorded;
- [ ] repeated channel trials are used if uncertainty is reported.
